# 04 - Joins y agregaciones (Gold)

## Objetivo
Construir `core_trips` (los 4 tipos, esquema común) y `economic_trips` (solo yellow/green/fhvhv, con fare/distancia/propina) a partir de la capa Silver, y calcular las 8 comparaciones que definimos, escribiendo cada resultado como Parquet pequeño en `gold_data/`.

**Prerrequisitos:**
- `03_etl_limpieza_spark.ipynb` ya corrido y validado (necesitamos `clean_data/` completo).
- Recordar: en `fhv`, `PULocationID`/`DOLocationID` usan `-1` como centinela de "ubicación desconocida" (81.93%/16.38% de los casos) — se excluyen solo en la comparación de demanda geográfica, no en las demás.

**Estrategia de rendimiento:** `core_trips` tiene ~770 millones de filas. En vez de volver a leerlo completo para cada una de las 8 comparaciones, calculamos primero **una sola tabla agregada base** (por tipo/fecha/hora) y de ahí derivamos 3 comparaciones sin tocar los datos crudos otra vez. Solo duración, tarifa/propina y demanda geográfica necesitan su propio pase sobre los datos completos.

**Salida esperada:** Parquet (y un JSON para la matriz de correlación) en `s3://xideralaws-curso-proyecto-alan/gold_data/`, uno por cada comparación.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as sf
import boto3
import json

BUCKET = "xideralaws-curso-proyecto-alan"

config = {
    "spark.jars.packages": "org.apache.hadoop:hadoop-aws:3.4.2,software.amazon.awssdk:bundle:2.29.52",
    "spark.hadoop.fs.s3a.aws.credentials.provider": "software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider",
    "spark.hadoop.fs.s3a.endpoint.region": "us-west-1",
    "spark.driver.memory": "2g"
}
spark = SparkSession.builder.appName("joinsAgregacionesSpark").config(map=config).getOrCreate()

s3_client = boto3.client("s3", region_name="us-west-1")
taxi_types = ["yellow", "green", "fhv", "fhvhv"]
tipos_economicos = ["yellow", "green", "fhvhv"]

:: loading settings :: url = jar:file:/home/ubuntu/opt/spark-4.1.2-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ubuntu/.ivy2.5.2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-604a5ca6-039d-4f19-abdd-73b5478bced6;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
:: resolution report :: resolve 423ms :: artifacts dl 16ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.4.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.1.4.Final from central in [default]
	software.amazon.awssdk#bundl

### Construir `core_trips` y `economic_trips` desde Silver
Ambas tablas se arman con `unionByName` sobre lo que ya dejó `03` en `clean_data/` — como los 4 tipos ya comparten exactamente los mismos nombres/tipos de columna (ese fue justo el propósito de normalizarlos en Silver), no hace falta `allowMissingColumns`.

In [2]:
core_cols = ["taxi_type", "pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID",
             "pickup_date", "pickup_hour", "day_of_week", "is_weekend", "duration_minutes"]

dfs_core = [
    spark.read.parquet(f"s3a://{BUCKET}/clean_data/{tipo}/year=*/month=*/").select(*core_cols)
    for tipo in taxi_types
]
core_trips = dfs_core[0]
for df in dfs_core[1:]:
    core_trips = core_trips.unionByName(df)

economic_cols = core_cols + ["distance", "fare", "tip"]
dfs_econ = [
    spark.read.parquet(f"s3a://{BUCKET}/clean_data/{tipo}/year=*/month=*/").select(*economic_cols)
    for tipo in tipos_economicos
]
economic_trips = dfs_econ[0]
for df in dfs_econ[1:]:
    economic_trips = economic_trips.unionByName(df)

print("core_trips y economic_trips construidos (lazy, todavia no se ha leido nada pesado)")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/18 17:25:14 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=*/month=*/.
java.io.FileNotFoundException: No such file or directory: s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=*/month=*
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOpera

core_trips y economic_trips construidos (lazy, todavia no se ha leido nada pesado)


### Tabla base agregada (un solo pase sobre `core_trips`)
Agrupamos por `taxi_type`, `pickup_date`, `pickup_hour` y `is_weekend` — esto reduce ~770M filas a un puñado de miles (4 tipos × ~3 años × 24 horas como máximo). La volvemos a leer justo después de escribirla para poder reutilizarla barata varias veces sin tocar `core_trips` de nuevo.

In [3]:
base_temporal = (core_trips
    .groupBy("taxi_type", "pickup_date", "pickup_hour", "is_weekend")
    .agg(sf.count(sf.lit(1)).alias("trips"))
)
base_temporal.write.mode("overwrite").parquet(f"s3a://{BUCKET}/gold_data/base_temporal/")

base_temporal = spark.read.parquet(f"s3a://{BUCKET}/gold_data/base_temporal/")
base_temporal.cache()
print("Filas en base_temporal:", base_temporal.count())

Filas en base_temporal: 85369


### Comparaciones 1, 6 y 7 — derivadas de `base_temporal` (sin releer datos crudos)
- **Volumen temporal** (1): suma de viajes por día y tipo.
- **Hora pico** (6): suma de viajes por hora y tipo (para el heatmap).
- **Weekday vs weekend** (7): suma de viajes agrupando por `is_weekend`.

In [4]:
trips_by_day = base_temporal.groupBy("taxi_type", "pickup_date").agg(sf.sum("trips").alias("trips"))
trips_by_day.write.mode("overwrite").parquet(f"s3a://{BUCKET}/gold_data/trips_by_day/")

peak_hours = base_temporal.groupBy("taxi_type", "pickup_hour").agg(sf.sum("trips").alias("trips"))
peak_hours.write.mode("overwrite").parquet(f"s3a://{BUCKET}/gold_data/peak_hours/")

weekday_weekend = base_temporal.groupBy("taxi_type", "is_weekend").agg(sf.sum("trips").alias("trips"))
weekday_weekend.write.mode("overwrite").parquet(f"s3a://{BUCKET}/gold_data/weekday_weekend/")

print("trips_by_day, peak_hours y weekday_weekend guardados")

trips_by_day, peak_hours y weekday_weekend guardados


### Comparación 2 — duración mediana por tipo
Este sí necesita un pase propio sobre `core_trips` completo: la mediana no se puede recalcular correctamente a partir de medianas parciales de `base_temporal` (promediar medianas no da la mediana real).

In [5]:
resultados_duracion = []
for tipo in taxi_types:
    df_tipo = spark.read.parquet(f"s3a://{BUCKET}/clean_data/{tipo}/year=*/month=*/").select("duration_minutes")
    resultado = (df_tipo
        .agg(sf.expr("percentile_approx(duration_minutes, 0.5)").alias("duracion_mediana_min"))
        .withColumn("taxi_type", sf.lit(tipo))
        .select("taxi_type", "duracion_mediana_min")
    )
    resultados_duracion.append(resultado)

trip_duration = resultados_duracion[0]
for r in resultados_duracion[1:]:
    trip_duration = trip_duration.unionByName(r)

trip_duration.write.mode("overwrite").parquet(f"s3a://{BUCKET}/gold_data/trip_duration/")
trip_duration.show()

26/09/18 17:27:38 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=*/month=*/.
java.io.FileNotFoundException: No such file or directory: s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=*/month=*
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:528)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:449)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAndSpan(S3AF

+---------+--------------------+
|taxi_type|duracion_mediana_min|
+---------+--------------------+
|   yellow|  13.316666666666666|
|    green|  12.333333333333334|
|      fhv|  18.433333333333334|
|    fhvhv|               16.15|
+---------+--------------------+



### Comparaciones 3 y 4 — tarifa por milla y propina % (solo yellow/green/fhvhv)
Usamos `economic_trips`, no `core_trips` — `fhv` no tiene estas columnas, así que ni siquiera aparece aquí (no es un dato faltante que haya que resolver, es una ausencia estructural del dataset original). Filtramos `distance`/`fare` > 0 para evitar divisiones inválidas.

In [6]:
resultados_fare = []
for tipo in tipos_economicos:
    df_tipo = (spark.read.parquet(f"s3a://{BUCKET}/clean_data/{tipo}/year=*/month=*/")
        .select("distance", "fare", "tip")
        .filter((sf.col("distance") > 0) & (sf.col("fare") > 0))
        .withColumn("fare_per_mile", sf.col("fare") / sf.col("distance"))
        .withColumn("tip_pct", (sf.col("tip") / sf.col("fare")) * 100)
        .agg(
            sf.expr("percentile_approx(fare_per_mile, 0.5)").alias("tarifa_por_milla_mediana"),
            sf.expr("percentile_approx(tip_pct, 0.5)").alias("propina_pct_mediana")
        )
        .withColumn("taxi_type", sf.lit(tipo))
        .select("taxi_type", "tarifa_por_milla_mediana", "propina_pct_mediana")
    )
    resultados_fare.append(df_tipo)

fare_and_tip = resultados_fare[0]
for r in resultados_fare[1:]:
    fare_and_tip = fare_and_tip.unionByName(r)

fare_and_tip.write.mode("overwrite").parquet(f"s3a://{BUCKET}/gold_data/fare_and_tip/")
fare_and_tip.show()

26/09/18 17:38:44 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=*/month=*/.
java.io.FileNotFoundException: No such file or directory: s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=*/month=*
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:528)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:449)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAndSpan(S3AF

+---------+------------------------+-------------------+
|taxi_type|tarifa_por_milla_mediana|propina_pct_mediana|
+---------+------------------------+-------------------+
|   yellow|       7.351219512195123| 22.598870056497177|
|    green|       6.791666666666667|  20.13422818791946|
|    fhvhv|       6.427135678391959|                0.0|
+---------+------------------------+-------------------+



### Comparación 5 — demanda geográfica
Aquí sí filtramos el centinela: `PULocationID != -1` excluye el 81.93% de `fhv` que no reporta ubicación. Es una limitación real del dataset, no algo que se pueda arreglar — se documentará así en el reporte final.

In [7]:
resultados_geo = []
for tipo in taxi_types:
    df_tipo = (spark.read.parquet(f"s3a://{BUCKET}/clean_data/{tipo}/year=*/month=*/")
        .select("PULocationID")
        .filter(sf.col("PULocationID") != -1)
        .groupBy("PULocationID")
        .agg(sf.count(sf.lit(1)).alias("trips"))
        .withColumn("taxi_type", sf.lit(tipo))
        .select("taxi_type", "PULocationID", "trips")
    )
    resultados_geo.append(df_tipo)

geographic_demand = resultados_geo[0]
for r in resultados_geo[1:]:
    geographic_demand = geographic_demand.unionByName(r)

geographic_demand.write.mode("overwrite").parquet(f"s3a://{BUCKET}/gold_data/geographic_demand/")
print("geographic_demand guardado (excluye ubicaciones desconocidas)")

26/09/18 17:54:42 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=*/month=*/.
java.io.FileNotFoundException: No such file or directory: s3a://xideralaws-curso-proyecto-alan/clean_data/yellow/year=*/month=*
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:528)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:449)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAndSpan(S3AF

geographic_demand guardado (excluye ubicaciones desconocidas)


### Comparación 8 — correlación cruzada de demanda
A partir de `base_temporal` (ya chico y cacheado), pivoteamos a viajes-por-día-por-tipo y calculamos la matriz de correlación con pandas (la tabla resultante es de a lo más ~1000 filas, perfectamente segura de colectar).

**Importante para el reporte:** esto mide asociación estadística entre los patrones de demanda de los 4 servicios, no causalidad — dos servicios pueden subir juntos simplemente porque ambos responden a la misma demanda general de movilidad en la ciudad (fin de semana, clima, eventos), no porque uno cause al otro.

In [8]:
conteo_diario_por_tipo = (base_temporal
    .groupBy("pickup_date")
    .pivot("taxi_type")
    .agg(sf.sum("trips"))
    .fillna(0)
    .orderBy("pickup_date")
)

pdf = conteo_diario_por_tipo.toPandas()
matriz_correlacion = pdf.drop(columns=["pickup_date"]).corr()
print(matriz_correlacion)

s3_client.put_object(
    Bucket=BUCKET,
    Key="gold_data/cross_taxi_correlation/correlation_matrix.json",
    Body=matriz_correlacion.to_json(indent=2).encode("utf-8"),
    ContentType="application/json"
)
print("\nMatriz de correlacion guardada en gold_data/cross_taxi_correlation/")

             fhv     fhvhv     green    yellow
fhv     1.000000  0.082132  0.371928  0.572614
fhvhv   0.082132  1.000000  0.251802  0.505550
green   0.371928  0.251802  1.000000  0.459701
yellow  0.572614  0.505550  0.459701  1.000000

Matriz de correlacion guardada en gold_data/cross_taxi_correlation/


### Verificación
Lista lo que quedó escrito en `gold_data/` — deberías ver una carpeta por cada comparación.

In [9]:
respuesta = s3_client.list_objects_v2(Bucket=BUCKET, Prefix="gold_data/", Delimiter="/")
for prefix in respuesta.get("CommonPrefixes", []):
    print(prefix["Prefix"])

gold_data/base_temporal/
gold_data/cross_taxi_correlation/
gold_data/fare_and_tip/
gold_data/geographic_demand/
gold_data/peak_hours/
gold_data/trip_duration/
gold_data/trips_by_day/
gold_data/weekday_weekend/


### Liberar memoria
Cierra la SparkSession antes de abrir `05_export_rds.ipynb`.

In [10]:
base_temporal.unpersist()
spark.stop()